# BDCC Midterm Review Notebook
**Companion to the BDCC Midterm Reviewer doc**

How to use this notebook:
1. For each **Practice** cell, replace `# YOUR CODE HERE` with your own implementation, from the algorithm, not from memory of your solved exercises.
2. Run the assert cell right after it. If it passes silently, you're correct.
3. If you get stuck, work the "by hand" markdown cell above it with pen and paper first, then translate your hand steps into code.
4. All toy datasets here are new (not the ones from your solved exercises), so this checks understanding, not recall.
5. Full worked solutions are in the **Solutions Appendix** at the very end — try each problem for real before peeking.


In [ ]:
import numpy as np
import pandas as pd
from itertools import combinations
from numpy.testing import (assert_equal, assert_almost_equal,
                            assert_array_equal, assert_array_almost_equal)


---
# Part A: Frequent Itemset Mining & Association Rule Mining

### Toy database
A small grocery basket dataset (5 transactions), different from your lecture exercise data.


In [ ]:
db = [
    {"chips", "salsa", "soda"},
    {"chips", "soda"},
    {"salsa", "guac", "chips"},
    {"soda", "guac"},
    {"chips", "salsa", "soda", "guac"},
]


### By hand first
Before coding, count support manually:
- How many transactions contain `chips`? `salsa`? `soda`? `guac`?
- With `minsup = 3`, which single items are frequent?
- Which 2-itemsets can you build from the frequent 1-itemsets, and what are their supports?

Write your tally on paper, then check it against `brute` below.


### Practice 1 — `to_sparse`
Return a sparse (or dense-for-simplicity) representation isn't required for the exam if you're short on time,
but being able to enumerate the universe of items sorted alphabetically is a building block for every function below.
Write a function that returns the sorted universe of items in `db`.

In [ ]:
def get_universe(db):
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
assert_equal(get_universe(db), ["chips", "guac", "salsa", "soda"])


### Practice 2 — `brute`
Create `brute(db, minsup)` returning frequent itemsets as a list of `(tuple_of_items, support)`,
items sorted alphabetically within each tuple, results sorted by decreasing support, then decreasing
length, then alphabetically. Use only the Python Standard Library.

In [ ]:
def brute(db, minsup):
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
fi = brute(db, 3)
assert_equal(len(fi), 6)
assert_equal(fi[0], (("chips",), 4))
assert_equal(fi[1], (("soda",), 4))


### Practice 3 — `apriori`
Same contract as `brute`, but implemented level-wise with candidate pruning (downward closure).
Should return identical results to `brute`, just faster on larger data.

In [ ]:
def apriori(db, minsup):
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
fi_apriori = apriori(db, 3)
assert_equal(fi_apriori, brute(db, 3))


### Practice 4 — maximal frequent itemsets
Create `maximal(fi)` that accepts the list of frequent itemsets (from `brute`/`apriori`) and
returns only the maximal ones (no frequent superset exists among the results), sorted the same
way as before.

In [ ]:
def maximal(fi):
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
maxfi = maximal(apriori(db, 3))
assert_equal(len(maxfi), 3)


### Practice 5 — `fpgrowth`
Implement FP-Growth via projected databases (pattern growth), same contract as `brute`/`apriori`.

In [ ]:
def fpgrowth(db, minsup):
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
assert_equal(fpgrowth(db, 3), brute(db, 3))


### Practice 6 — `assoc`
Create `assoc(fi, minconf, db_size)` returning association rules with confidence >= `minconf`
as a list of dicts with keys `antecedent`, `consequent`, `support`, `confidence`, `lift`.
Sort by decreasing lift, then decreasing confidence, then decreasing support.

In [ ]:
def assoc(fi, minconf, db_size):
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
rules = assoc(apriori(db, 3), 0.6, len(db))
assert len(rules) >= 1
# every rule must clear the confidence bar
assert all(r["confidence"] >= 0.6 for r in rules)


---
# Part B: Neighborhood-based Collaborative Filtering

### Toy utility matrix
Ratings 1-5, blanks unrated. Different numbers from your lecture exercise.


In [ ]:
df_util = pd.DataFrame(
    {
        "ItemA": [5, 4, np.nan, 2],
        "ItemB": [3, np.nan, 4, 1],
        "ItemC": [4, 5, 2, np.nan],
        "ItemD": [np.nan, 3, 5, 3],
    },
    index=["U1", "U2", "U3", "U4"],
)
df_util


### By hand first
Compute each user's mean rating. Then, for U1 and U2, mean-center their ratings on the items they
BOTH rated (ItemA, ItemC), and compute the adjusted cosine similarity by hand before checking your
code.

### Practice 1 — `raw_cos`
Cosine similarity between two pandas Series, using only positions where both are non-null.
Return 0 if there's no overlap or a zero-magnitude vector.

In [ ]:
def raw_cos(a, b):
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
assert_almost_equal(raw_cos(df_util.loc["U1"], df_util.loc["U2"]), 0.975609756097561)
assert_equal(raw_cos(pd.Series([np.nan]), pd.Series([np.nan])), 0)


### Practice 2 — `user_complete`
Mean-center the matrix, then use adjusted cosine similarity (raw_cos on the mean-centered rows) to
complete the matrix via user-based CF with up to `k` neighbors. Ties broken by user/item id.

In [ ]:
def user_complete(df_utility, k):
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
completed = user_complete(df_util, 2)
assert_equal(completed.shape, df_util.shape)
# originally-rated cells must be unchanged
assert_almost_equal(completed.loc["U1", "ItemA"], 5)


### Practice 3 — `item_complete`
Same idea as `user_complete`, but find similar items (columns) instead of similar users.

In [ ]:
def item_complete(df_utility, k):
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
completed_item = item_complete(df_util, 2)
assert_equal(completed_item.shape, df_util.shape)


### Practice 4 — `recommend`
Given the original and completed utility matrices, return the top `n` unrated items for a user,
sorted by predicted rating descending then item id ascending.

In [ ]:
def recommend(user, df_utility, df_completed, n):
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
recos = recommend("U1", df_util, completed, 1)
assert_equal(len(recos), 1)
assert "ItemD" in recos  # U1's only unrated item, must be included


### Practice 5 — `dcg` and `ndcg`
`dcg(df_utility, recommendations)`: mean DCG across the given users' recommendation lists.
`ndcg(df_utility, recommendations)`: DCG divided by the ideal DCG for those users.

In [ ]:
def dcg(df_utility, recommendations):
    # YOUR CODE HERE
    raise NotImplementedError()

def ndcg(df_utility, recommendations):
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
# a perfect ranking (already sorted by true rating, descending) should give NDCG = 1
perfect = [df_util.loc["U2"].dropna().sort_values(ascending=False).index.tolist()]
assert_almost_equal(ndcg(df_util.loc[["U2"]], perfect), 1.0)


---
# Part C: Latent Factor-based Collaborative Filtering

We'll reuse `df_util` from Part B.

### By hand first
With `d = 1` latent factor and both U and V initialized to all ones, work out (on paper) what the
first coordinate-descent update for `v` of ItemA would be, using only the ratings that exist for
ItemA (U1 = 5, U2 = 4). Compare it against the Reviewer doc's worked example before coding.

### Practice 1 — `als`
Alternating least squares: accepts the utility matrix, number of latent factors `d`, and tolerance
`tol`; returns `(F_user, F_item)` once RMSE on the known entries is below `tol`. Initialize both
matrices to all ones.

In [ ]:
def als(M, d, tol):
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
f_user, f_item = als(df_util.to_numpy(), 2, 0.5)
assert_equal(f_user.shape, (4, 2))
assert_equal(f_item.shape, (4, 2))
recon = f_user @ f_item.T
mask = np.isfinite(df_util.to_numpy())
rmse = np.sqrt(np.nanmean((df_util.to_numpy() - recon)[mask] ** 2))
assert rmse < 0.5


### Practice 2 — `recommend_als`
Return the top `L` unrated items for a user (by row position) using the ALS factors, sorted by
predicted rating descending then item id ascending.

In [ ]:
def recommend_als(user, df_utility, f_user, f_item, L):
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
recos_als = recommend_als(0, df_util, f_user, f_item, 1)
assert_equal(recos_als, ["ItemD"])  # U1's only unrated item


### Practice 3 — `cd` (coordinate descent)
Accepts the utility matrix, number of latent factors `k`, and tolerance `tol` (stop when the
fractional SSE improvement drops below `tol`). Initialize both matrices to all ones, update U
row-by-row then V row-by-row, one coordinate at a time.

In [ ]:
def cd(df_utility, k, tol):
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
f_user_cd, f_item_cd = cd(df_util, 2, 0.01)
assert_equal(f_user_cd.shape, (4, 2))
assert_equal(f_item_cd.shape, (4, 2))


### Practice 4 — `recommend_cd`
Same contract as `recommend_als`, but using the coordinate-descent factors.

In [ ]:
def recommend_cd(user, df_utility, f_user, f_item, N):
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
recos_cd = recommend_cd(3, df_util, f_user_cd, f_item_cd, 1)
assert_equal(recos_cd, ["ItemC"])  # U4's only unrated item


---
# Part D: Mini mock exam (end-to-end, like Assignment 1)

This mirrors the shape of a likely hands-on exam item: run FIM + ARM on a transaction dataset,
then use a CF method on a ratings dataset, and interpret both for a business question.

### Setup: a slightly bigger toy transaction dataset (a small café's orders)


In [ ]:
orders = [
    {"latte", "croissant"},
    {"latte", "muffin"},
    {"espresso", "croissant"},
    {"latte", "croissant", "muffin"},
    {"espresso", "muffin"},
    {"latte", "croissant"},
    {"latte", "espresso", "croissant"},
    {"muffin"},
]


**Task 1.** Using your `fpgrowth` and `assoc` functions above, find frequent itemsets with
`minsup = 3` and association rules with `minconf = 0.5`. What's the highest-lift rule? YOUR CODE HERE.

In [ ]:
# YOUR CODE HERE


**Task 2.** Write 1-2 sentences: what would you recommend to the café owner based on the
rule you found (e.g., a combo deal, menu placement)?

_(Your answer here.)_

**Task 3.** Using `df_util` from Part B/C, compute recommendations for every user with
`user_complete` (k=2) and with `als` (d=2). Do the top recommendation agree for any user? If they
disagree, which would you trust more and why?

In [ ]:
# YOUR CODE HERE


_(Your answer here.)_

---
# Solutions Appendix
**Only look here after attempting every Practice cell above on your own.**
These are reference implementations — there's more than one correct way to write each function;
what matters for the exam is that your version passes the asserts and you understand *why* it works.


### Solution — get_universe

In [ ]:
def get_universe(db):
    return sorted({item for t in db for item in t})


### Solution — brute

In [ ]:
def brute(db, minsup):
    universe = sorted({item for t in db for item in t})
    itemsets = {c: 0 for r in range(1, len(universe) + 1) for c in combinations(universe, r)}
    for itemset in itemsets:
        for t in db:
            if all(item in t for item in itemset):
                itemsets[itemset] += 1
    fi = [(iset, sup) for iset, sup in itemsets.items() if sup >= minsup]
    return sorted(fi, key=lambda x: (-x[1], -len(x[0]), x[0]))


### Solution — apriori

In [ ]:
def apriori(db, minsup):
    universe = sorted({item for t in db for item in t})
    k = 1
    results = []
    candidates = {c: 0 for c in combinations(universe, k)}
    while candidates:
        for c in candidates:
            for t in db:
                if all(item in t for item in c):
                    candidates[c] += 1
        frequent = [(c, sup) for c, sup in candidates.items() if sup >= minsup]
        results.extend(frequent)
        k += 1
        items_left = sorted({item for iset, sup in frequent for item in iset})
        candidates = {c: 0 for c in combinations(items_left, k)}
    return sorted(results, key=lambda x: (-x[1], -len(x[0]), x[0]))


### Solution — maximal

In [ ]:
def maximal(fi):
    sets = [set(iset) for iset, sup in fi]
    out = [fi[i] for i, s in enumerate(sets)
           if not any(s < other for j, other in enumerate(sets) if j != i)]
    return sorted(out, key=lambda x: (-x[1], -len(x[0]), x[0]))


### Solution — fpgrowth

In [ ]:
def fpgrowth(db, minsup):
    def grow(D, prefix, minsup):
        universe = sorted({item for t in D for item in t})
        counts = {item: 0 for item in universe}
        for t in D:
            for item in t:
                counts[item] += 1
        frequent_items = [item for item in universe if counts[item] >= minsup]
        results = []
        for z in frequent_items:
            new_prefix = tuple(sorted(prefix + (z,)))
            results.append((new_prefix, counts[z]))
            projected = [{item for item in t if item > z} for t in D if z in t]
            results.extend(grow(projected, new_prefix, minsup))
        return results
    fi = grow(db, (), minsup)
    return sorted(fi, key=lambda x: (-x[1], -len(x[0]), x[0]))


### Solution — assoc

In [ ]:
def assoc(fi, minconf, db_size):
    support = {iset: sup for iset, sup in fi}
    rules = []
    for iset, sup in fi:
        if len(iset) < 2:
            continue
        for consequent in iset:
            ante = tuple(sorted(set(iset) - {consequent}))
            ante_sup = support[ante]
            cons_sup = support[(consequent,)]
            confidence = sup / ante_sup
            if confidence < minconf:
                continue
            lift = confidence / (cons_sup / db_size)
            rules.append({
                "antecedent": ante if len(ante) > 1 else ante[0],
                "consequent": consequent,
                "support": sup,
                "confidence": confidence,
                "lift": lift,
            })
    return sorted(rules, key=lambda r: (-r["lift"], -r["confidence"], -r["support"]))


### Solution — raw_cos

In [ ]:
def raw_cos(a, b):
    mask = a.notna() & b.notna()
    a, b = a[mask].to_numpy(), b[mask].to_numpy()
    if len(a) == 0:
        return 0
    mag_a, mag_b = np.linalg.norm(a), np.linalg.norm(b)
    if mag_a == 0 or mag_b == 0:
        return 0
    return (a @ b) / (mag_a * mag_b)


### Solution — user_complete

In [ ]:
def user_complete(df_utility, k):
    centered = df_utility.sub(df_utility.mean(axis=1), axis=0)
    completed = df_utility.copy()
    for user in df_utility.index:
        sims = sorted(
            [(raw_cos(centered.loc[user], centered.loc[other]), other)
             for other in df_utility.index if other != user],
            key=lambda x: (-x[0], x[1]),
        )[:k]
        for item in df_utility.columns:
            if pd.notna(df_utility.loc[user, item]):
                continue
            rated = [(s, o) for s, o in sims if pd.notna(df_utility.loc[o, item])]
            if not rated:
                continue
            num = sum(s * centered.loc[o, item] for s, o in rated)
            den = sum(abs(s) for s, o in rated)
            if den == 0:
                continue
            completed.loc[user, item] = df_utility.loc[user].mean() + num / den
    return completed


### Solution — item_complete

In [ ]:
def item_complete(df_utility, k):
    centered = df_utility.sub(df_utility.mean(axis=1), axis=0)
    completed = df_utility.copy()
    for item in df_utility.columns:
        sims = sorted(
            [(raw_cos(centered[item], centered[other]), other)
             for other in df_utility.columns if other != item],
            key=lambda x: (-x[0], x[1]),
        )[:k]
        for user in df_utility.index:
            if pd.notna(df_utility.loc[user, item]):
                continue
            rated = [(s, o) for s, o in sims if pd.notna(df_utility.loc[user, o])]
            if not rated:
                continue
            num = sum(s * centered.loc[user, o] for s, o in rated)
            den = sum(abs(s) for s, o in rated)
            if den == 0:
                continue
            completed.loc[user, item] = df_utility.loc[user].mean() + num / den
    return completed


### Solution — recommend

In [ ]:
def recommend(user, df_utility, df_completed, n):
    unrated = df_utility.loc[user].isna()
    preds = df_completed.loc[user, unrated].dropna()
    ranked = sorted(preds.items(), key=lambda x: (-x[1], x[0]))
    return [item for item, _ in ranked[:n]]


### Solution — dcg / ndcg

In [ ]:
def dcg(df_utility, recommendations):
    total = 0
    for user, items in zip(df_utility.index, recommendations):
        for rank, item in enumerate(items, start=1):
            rating = df_utility.loc[user, item]
            if pd.notna(rating):
                total += (2 ** rating - 1) / np.log2(rank + 1)
    return total / len(recommendations)

def ndcg(df_utility, recommendations):
    ideal = [
        df_utility.loc[user].dropna().sort_values(ascending=False).index[:len(items)].tolist()
        for user, items in zip(df_utility.index, recommendations)
    ]
    return dcg(df_utility, recommendations) / dcg(df_utility, ideal)


### Solution — als

In [ ]:
def als(M, d, tol):
    from scipy.linalg import lstsq
    M = np.asarray(M, dtype=float)
    U = np.ones((M.shape[0], d))
    V = np.ones((M.shape[1], d))
    while True:
        for j in range(V.shape[0]):
            has = np.isfinite(M[:, j])
            V[j, :] = lstsq(U[has, :], M[:, j][has])[0]
        for i in range(U.shape[0]):
            has = np.isfinite(M[i, :])
            U[i, :] = lstsq(V[has, :], M[i, :][has])[0]
        rmse = np.sqrt(np.nanmean((M - U @ V.T) ** 2))
        if rmse < tol:
            return U, V


### Solution — recommend_als / recommend_cd

In [ ]:
def recommend_als(user, df_utility, f_user, f_item, L):
    unrated_mask = df_utility.iloc[user].isna()
    unrated_items = df_utility.columns[unrated_mask]
    positions = np.where(unrated_mask.to_numpy())[0]
    preds = f_item[positions, :] @ f_user[user, :]
    ranked = sorted(zip(unrated_items, preds), key=lambda x: (-x[1], x[0]))
    return [item for item, _ in ranked[:L]]

recommend_cd = recommend_als  # identical logic once you have factor matrices


### Solution — cd (coordinate descent)

In [ ]:
def cd(df_utility, k, tol):
    M = np.asarray(df_utility, dtype=float)
    mask = np.isfinite(M)
    n, m = M.shape
    U = np.ones((n, k))
    V = np.ones((m, k))

    def sse(U, V):
        err = np.where(mask, M - U @ V.T, 0.0)
        return np.sum(err ** 2)

    prev = sse(U, V)
    while True:
        for i in range(n):
            obs = mask[i, :]
            if not obs.any():
                continue
            Vobs, Mobs = V[obs, :], M[i, obs]
            pred = U[i, :] @ Vobs.T
            for f in range(k):
                Vf = Vobs[:, f]
                denom = Vf @ Vf
                if denom == 0:
                    continue
                resid = Mobs - pred + U[i, f] * Vf
                U[i, f] = (Vf @ resid) / denom
                pred = U[i, :] @ Vobs.T
        for j in range(m):
            obs = mask[:, j]
            if not obs.any():
                continue
            Uobs, Mobs = U[obs, :], M[obs, j]
            pred = Uobs @ V[j, :]
            for f in range(k):
                Uf = Uobs[:, f]
                denom = Uf @ Uf
                if denom == 0:
                    continue
                resid = Mobs - pred + V[j, f] * Uf
                V[j, f] = (Uf @ resid) / denom
                pred = Uobs @ V[j, :]
        cur = sse(U, V)
        if prev == 0 or abs(prev - cur) / prev < tol:
            return U, V
        prev = cur
